# Your own Python setup: environments and pip

Python stage, lesson P24. Part of [datafromzero](https://datafromzero.github.io), data and text CC BY 4.0.

**Goals:** make an environment, install an exact version into the Python you name, record it in `requirements.txt`, and print the versions a notebook runs on.

**How to use this notebook:** run the cells from top to bottom. When in doubt, **Run all from the top**: restart the session and run every cell from the first.
Every exercise has a cell for your code and a check cell that compares your answer with a known answer from the cafe-sales data. Hints and solutions are folded away
under each exercise, free to open. The working cell of Exercise 2 starts empty: you write it.

## Setup

In [ ]:
import math
import numbers

import numpy as np
import pandas as pd

# Run this cell as it is. It also holds the answer checker used by the exercise cells.
EXERCISES = ["Exercise 1", "Exercise 2", "Exercise 3"]


RESULTS = {}


def same(got, expected):
    """True when two values agree: floats within rounding, lists and dicts item by item."""
    try:
        if isinstance(expected, dict):
            return isinstance(got, dict) and got.keys() == expected.keys() and all(same(got[k], expected[k]) for k in expected)
        if isinstance(expected, (list, tuple)):
            return isinstance(got, (list, tuple)) and len(got) == len(expected) and all(same(g, e) for g, e in zip(got, expected))
        if isinstance(expected, bool) or isinstance(got, bool):
            return type(got) is type(expected) and got == expected
        if isinstance(expected, float):
            return isinstance(got, numbers.Real) and math.isclose(got, expected, rel_tol=1e-9, abs_tol=1e-9)
        if isinstance(expected, int):
            return isinstance(got, numbers.Integral) and got == expected
        return type(got) is type(expected) and got == expected
    except Exception:
        return False


def check(label, name, expected, traps=None):
    """Compare the variable called `name` with the known answer. It never stops Run all."""
    got = globals().get(name)
    if got is None:
        RESULTS[label] = None
        print(f"{label}: give {name} a value, then run this cell again.")
        return False
    if same(got, expected):
        RESULTS[label] = True
        print(f"{label}: correct.")
        return True
    RESULTS[label] = False
    print(f"{label}: not yet. Expected {expected!r}, but {name} is {got!r}.")
    for wrong, message in (traps or {}).items():
        if same(got, wrong):
            print("Hint:", message)
    return False

def check_function(label, name, cases):
    """Call your function on every case. Each case is (arguments, keyword arguments, expected result)."""
    fn = globals().get(name)
    if not callable(fn):
        RESULTS[label] = None
        print(f"{label}: define a function called {name}, then run this cell again.")
        return False
    for args, kwargs, expected in cases:
        try:
            got = fn(*args, **kwargs)
        except Exception as error:
            RESULTS[label] = False
            print(f"{label}: not yet. {name}{args} stopped with {type(error).__name__}: {error}")
            return False
        if not same(got, expected):
            RESULTS[label] = False
            print(f"{label}: not yet. {name}{args} should give {expected!r}, but gave {got!r}.")
            if got is None:
                print("Hint: None means the function gave nothing back. Did it return the value, or only print it?")
            return False
    RESULTS[label] = True
    print(f"{label}: correct ({len(cases)} cases).")
    return True

## Data

In [ ]:
# The five cafe-sales tables (version 1), read from the release tag v0.1.0.
BASE = "https://raw.githubusercontent.com/datafromzero/datafromzero.github.io/v0.1.0/datasets/cafe-sales/v1/"
stores = pd.read_csv(BASE + "stores.csv")
customers = pd.read_csv(BASE + "customers.csv")
products = pd.read_csv(BASE + "products.csv")
orders = pd.read_csv(BASE + "orders.csv")
order_items = pd.read_csv(BASE + "order_items.csv")

# Known answer from the dataset README: 3, 150, 16, 1,800 and 3,287 rows.
assert (len(stores), len(customers), len(products), len(orders), len(order_items)) == (3, 150, 16, 1800, 3287)
print("Data loaded: 3 stores, 150 customers, 16 products, 1,800 orders, 3,287 order lines.")
# Helpers for this lesson: run this cell as it is. Unit 1 explains what they do.
import os
import subprocess
import sys
import tempfile
from pathlib import Path

os.environ["PIP_DISABLE_PIP_VERSION_CHECK"] = "1"  # keeps pip from printing its update notice
os.environ["NO_COLOR"] = "1"  # plain text in the messages we print
os.environ["PYTHON_COLORS"] = "0"
work = Path(tempfile.mkdtemp())  # a scratch folder for the environments of this lesson
env_dir = work / "cafe-env"
env_python = env_dir / ("Scripts/python.exe" if sys.platform == "win32" else "bin/python")  # the Python inside an environment


def run_in(python, code):
    """Run a few lines with the Python at `python` and give back the last line it printed, error included."""
    done = subprocess.run([str(python), "-c", code], capture_output=True, text=True)
    lines = (done.stdout + done.stderr).strip().splitlines()
    return lines[-1] if lines else "(nothing printed)"


def make_env(name):
    """Make a new, empty environment called `name` and give back the path of its Python."""
    folder = work / name
    subprocess.run([sys.executable, "-m", "venv", str(folder)])
    return folder / ("Scripts/python.exe" if sys.platform == "win32" else "bin/python")

## Unit 1: An environment is a folder with its own Python

**Predict first.** this notebook has pandas. Will the Python of the new folder find it, and what is the last line it prints?

In [ ]:
!{sys.executable} -m venv {env_dir}
print(run_in(env_python, "import pandas"))

## Unit 2: pip installs into the Python you name

**Predict first.** after this install, will the notebook's own version change, and what will `cafe-env` print?

In [ ]:
!{env_python} -m pip install -q pandas==2.2.3
print("notebook:", pd.__version__)
print("cafe-env:", run_in(env_python, "import pandas; print(pandas.__version__)"))

**Predict first.** which version will the notebook report, and which of the two Pythons got the package?

In [ ]:
%pip install -q tabulate==0.9.0
import tabulate
print(tabulate.__version__)

## Unit 3: requirements.txt: write down what you pinned

**Predict first.** the file names 2 packages. Will `pip freeze` print more than 2 lines, and which lines hold the two pins?

In [ ]:
requirements = work / "requirements.txt"
requirements.write_text("pandas==2.2.3\ntabulate==0.9.0\n")
!{env_python} -m pip install -q -r {requirements}
frozen = !{env_python} -m pip freeze
print(len(frozen) > 2)
print([line for line in frozen if line.startswith(("pandas", "tabulate"))])

## Unit 4: Print the versions: one file, two Pythons

**Predict first.** the file and the call are the same in both. Will the two lines print the same row count, and the same type for `status`?

In [ ]:
print("python:", sys.version_info[:2])
orders_in_env = f'import pandas as pd; orders = pd.read_csv("{BASE}orders.csv"); print(len(orders), orders["status"].dtype)'
print("notebook:", len(orders), orders["status"].dtype)
print("cafe-env:", run_in(env_python, orders_in_env))

## Finish it

### Exercise 1: Finish it

**Task:** A colleague set up `second-env` for the Harbor manager and installed `six` 1.17.0 for her. Does `second-env` have it? The cell makes `second-env`, a new empty environment, runs the colleague's install line, and asks `second-env` for its `six` version. Read the install line and say what you expect `answer_1` to be, then run the cell. Finish it: change the install line so that the answer is the version that was asked for.

**Expected:** `answer_1` is `1.17.0`.

In [ ]:
second_python = make_env("second-env")  # given: a new, empty environment
!{sys.executable} -m pip install -q six==1.17.0  # the colleague's install line
answer_1 = run_in(second_python, "import six; print(six.__version__)")  # given: ask second-env

In [ ]:
_ = check("Exercise 1", "answer_1", "1.17.0", traps={"ModuleNotFoundError: No module named 'six'": "The package went into a different Python. Which Python does the install line name?"})

<details>
<summary>Hint</summary>

The Python in front of `-m pip` decides where the package goes. The cell already holds a variable with the path of the Python of `second-env`.

</details>

<details>
<summary>Solution</summary>

```python
second_python = make_env("second-env")
!{second_python} -m pip install -q six==1.17.0
answer_1 = run_in(second_python, "import six; print(six.__version__)")
```

`second-env` prints `1.17.0`. The colleague's line named `sys.executable`, the Python of the notebook, so `six` went there and `second-env` stayed empty; the check cell reports the error that `second-env` prints. `%pip` is no fix: it also installs into the notebook's own Python.

</details>

## On your own

### Exercise 2: On your own

**Task:** The cafe notebook needs pandas 2.2.3 or newer. How does a script know whether an installed version is new enough? Before you start, say in a sentence what `version_at_least("2.10.0", "2.9.1")` should give. Then write `version_at_least(installed, needed)`: it gives `True` when the first version is the same as or newer than the second. A version is text with three numbers separated by dots, such as `"2.2.3"`. Write the function in the empty cell of the notebook.

**Expected:** `version_at_least("2.10.0", "2.9.1")` gives `True` and `version_at_least("2.2.3", "3.0.0")` gives `False`.

In [ ]:
_ = check_function("Exercise 2", "version_at_least", [(("2.10.0", "2.9.1"), {}, True), (("2.2.3", "2.2.3"), {}, True), (("2.2.3", "3.0.0"), {}, False), (("3.1.1", "2.2.3"), {}, True), (("2.9.1", "2.10.0"), {}, False)])

<details>
<summary>Hint</summary>

Cut the text at the dots with `split(".")`, turn each piece into a whole number with `int`, and compare the two lists of numbers. A list compares item by item, from the first.

</details>

<details>
<summary>Solution</summary>

```python
def version_at_least(installed, needed):
    return [int(part) for part in installed.split(".")] >= [int(part) for part in needed.split(".")]
```

The function turns each version into a list such as `[2, 10, 0]` and lets Python compare the lists item by item, so 10 beats 9. Comparing the texts directly looks at one character at a time, and `"1"` comes before `"9"`, so `"2.10.0"` would count as older than `"2.9.1"`; the check cell calls it with that pair.

</details>

## Check yourself

### Exercise 3: Check yourself: spot the flaw

**Task:** Spot the flaw. A colleague wrote a check that lists the packages the cafe notebook imports but its `requirements.txt` does not name. It prints all four packages as missing, so nobody trusts it. Fix the check so that `missing` holds only the package that is really absent.

**Expected:** `missing` is `['seaborn']`.

In [ ]:
imports_used = ["pandas", "matplotlib", "seaborn", "numpy"]  # what the cafe notebook imports
colleague_file = "pandas==2.2.3\nmatplotlib==3.9.2\nnumpy==2.1.2\n"  # the requirements.txt, written by hand
listed = colleague_file.splitlines()  # the colleague's check
missing = [name for name in imports_used if name not in listed]

In [ ]:
_ = check("Exercise 3", "missing", ["seaborn"], traps={("pandas", "matplotlib", "seaborn", "numpy"): "Print `listed`. Each item is a whole line such as `pandas==2.2.3`, so it never equals a bare name."})

<details>
<summary>Hint</summary>

Print `listed` and read one item. Does any item equal the bare name `"pandas"`?

</details>

<details>
<summary>Solution</summary>

```python
imports_used = ["pandas", "matplotlib", "seaborn", "numpy"]
colleague_file = "pandas==2.2.3\nmatplotlib==3.9.2\nnumpy==2.1.2\n"
listed = [line.split("==")[0] for line in colleague_file.splitlines()]
missing = [name for name in imports_used if name not in listed]
```

`listed` held whole lines such as `pandas==2.2.3`, so no bare name was ever in it and every package looked missing. Cutting each line at `==` and keeping the first piece leaves the names, and only `seaborn` is absent. The fix handles `==` lines only: a real file can also hold comments, blank lines and other forms such as `>=`.

</details>

## Run all from the top

Restart the session, run every cell from the first, then read this cell last.

In [ ]:
for label in EXERCISES:
    state = RESULTS.get(label)
    print(f"{label}: " + ("correct" if state else "not yet" if state is False else "not started"))
print(f"{sum(1 for label in EXERCISES if RESULTS.get(label))} of {len(EXERCISES)} exercises correct.")